In [1]:
import seaborn as sns
import numpy as np
import sctop as top
import matplotlib.pyplot as plt
import scanpy as sc

Control data [GSE246441](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE246441)

In [3]:
control_data = sc.read_h5ad('/Users/maria/github/phenotype-signaling/GSE246441_all_10x_data.h5ad')

In [13]:
control_data.obs['cell_states'].value_counts()

cell_states
Basal            11316
Club              6755
Multiciliated     1096
Goblet             361
Rare                52
Name: count, dtype: int64

# Phenotype-signal model with consumer-resource-like behavior

$$
\frac{dN_\mu}{dt} = \sum_{\nu \alpha} T_{\mu \nu} c_{\nu \alpha} L_{\alpha} N_{\nu} - \sum_{\gamma \beta} T_{\gamma \mu} c_{\mu \beta} L_{\beta} N_{\mu} - m_{\mu} N_{\mu}
$$

$$
\frac{dL_{\alpha}}{dt} = k_{\alpha} - d L_{\alpha} - \sum_{\mu \nu} T_{\mu \nu} c_{\nu \alpha} N_{\mu} L_{\alpha} + \sum_{\gamma} P_{\gamma \beta} N_{\gamma} 
$$

Variables:
- $N_{\mu}$: number of cells of cell type $\mu$
- $L_{\alpha}$: number of ligands of type $\alpha$
- $T_{\mu \nu}$: probability of transitioning from cell type $\nu$ to cell type $\mu$
- $c_{\nu \alpha}$: responsivity of cell type $\nu$ to ligand $\alpha$
- $m_{\mu}$: "mortality" of cell type $\mu$ (cell turnover rate for a given cell type?)
- $k_{\alpha}$: external supply of ligand $\alpha$
- $d$: dissociation rate of ligands
- $P_{\gamma \beta}$: production of ligand $\beta$ by cell type $\gamma$

Parameters from gene expression:
- Use Hopfield-like proximity-based likelihood for cell-type transition matrix: $T_{\mu \nu} = \frac{e^{- \beta ||\xi_{\mu} - \xi_{\nu}||^2}}{\sum_{\gamma} e^{- \beta ||\xi_{\mu} - \xi_{\gamma}||^2}}$ where $\xi_{\mu}$ is the gene expression profile for cell type $\mu$
- Combine receptor availibility with information about ligand-binding to find the phenotype-specific ligand responsivity: $c_{\nu \alpha} = \sum_i^{N_r} \xi_{\nu i} B_{i \alpha}$ where $i$ is summing over the receptors and $B$ is a binary matrix indicating whether ligand $\alpha$ binds to receptor $i$
- Use gene expression of ligand $\beta$ in cell type $\gamma$ for $P_{\gamma \beta}$

Free parameters: $m_{\mu}$, $d$